# MeCab (fugashi + UniDic) にユーザー辞書を追加する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/mecab_userdic.ipynb)

NLP クックブック 03-17 ／ 日本語の形態素解析と文分割

UniDic で細かく切れてしまう専門用語や固有名詞を、MeCab のユーザー辞書に登録して 1 語にしたい。CSV に表層形・左右文脈 ID・生起コスト・品詞・読みを書き、`mecab-dict-index` に当たるコンパイラで `user.dic` にして `Tagger('-u user.dic')` で読み込む。fugashi の wheel にはコンパイラが同梱されているので、apt で MeCab を入れなくてよい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
from fugashi import Tagger

tagger = Tagger()
# 各語の左文脈 ID (%phl)・右文脈 ID (%phr)・生起コスト (%pw) と品詞を出す出力形式
info = Tagger(r'-O "" -F "%m\t%phl\t%phr\t%pw\t%f[0],%f[1],%f[2]\n" -U "%m\t%phl\t%phr\t%pw\t未知語\n"')

sents = ["大規模言語モデルで検索拡張生成を試す。", "外国人参政権について話し合った。",
         "形態素解析にフガシを使う。", "都立大に行く。"]
for s in sents:
    print(" | ".join(w.surface for w in tagger(s)))
print()
print(info.parse("大規模言語モデルを東大で試す"))

大 | 規模 | 言語 | モデル | で | 検索 | 拡張 | 生成 | を | 試す | 。
外国 | 人参 | 政権 | に | つい | て | 話し合っ | た | 。
形態 | 素 | 解析 | に | フ | ガシ | を | 使う | 。
都立 | 大 | に | 行く | 。

大	5957	5957	8306	接頭辞,,
規模	5146	5146	7386	名詞,普通名詞,一般
言語	5146	5146	3727	名詞,普通名詞,一般
モデル	5144	5144	646	名詞,普通名詞,一般
を	806	806	4464	助詞,格助詞,
東大	4786	4786	8591	名詞,固有名詞,一般
で	789	789	4934	助詞,格助詞,
試す	1327	1327	9622	動詞,一般,
EOS


In [3]:
import csv, subprocess, sys
import unidic_lite

def ref_entry(surface, kana, ref, cost=None):
    """似た既存語 ref から左右文脈 ID・生起コスト・品詞を借りて、ユーザー辞書の CSV の 1 行を作る"""
    lid, rid, ref_cost = info.parse(ref).split("\n")[0].split("\t")[1:4]
    f = tagger(ref)[0].feature
    # unidic-lite の素性は 26 列（dicrc の先頭に一覧がある）。品詞・活用・語種は ref のものを使い、表記と読みを差し替える。
    # 語彙素読み・発音・アクセントなどは簡単のため読みか * で埋める
    feats = [f.pos1, f.pos2, f.pos3, f.pos4, f.cType, f.cForm,
             kana, surface, surface, kana, surface, kana, f.goshu, "*", "*", "*", "*",
             kana, kana, kana, kana, "*", "*", "*", "*", "*"]
    return [surface, lid, rid, str(ref_cost if cost is None else cost)] + feats

def build_userdic(rows, name):
    """CSV を書き、fugashi 同梱のコンパイラ（mecab-dict-index と同じもの）で name.dic にする"""
    with open(f"{name}.csv", "w", newline="", encoding="utf-8") as fp:
        csv.writer(fp).writerows(rows)
    # build_dictionary は失敗するとプロセスごと終了するので、カーネルを巻き込まないよう別プロセスで呼ぶ
    code = "import sys, fugashi; fugashi.build_dictionary('mecab-dict-index ' + ' '.join(sys.argv[1:]))"
    r = subprocess.run([sys.executable, "-c", code, "-d", unidic_lite.DICDIR, "-u", f"{name}.dic",
                        "-f", "utf-8", "-t", "utf-8", f"{name}.csv"], capture_output=True, text=True)
    assert r.returncode == 0, r.stdout + r.stderr
    return Tagger(f"-u {name}.dic")

entries = [  # (表層形, 読み, 品詞・ID・コストを借りる既存語)
    ("大規模言語モデル", "ダイキボゲンゴモデル", "言語"),
    ("検索拡張生成", "ケンサクカクチョウセイセイ", "生成"),
    ("参政権", "サンセイケン", "政権"),
    ("フガシ", "フガシ", "グーグル"),
    ("都立大", "トリツダイ", "東大"),
]
rows = [ref_entry(*e) for e in entries]
for row in rows:
    print(",".join(row[:10]), "...", f"({len(row)} 列)")
user = build_userdic(rows, "user")
print()
for s in sents:
    print("前:", " | ".join(w.surface for w in tagger(s)))
    print("後:", " | ".join(w.surface for w in user(s)))
print()
for w in user("形態素解析にフガシを使う。"):
    f = w.feature
    print(w.surface, f.pos1, f.pos2, f.pos3, f.kana, w.is_unk, sep="\t")

大規模言語モデル,5146,5146,3727,名詞,普通名詞,一般,*,*,* ... (30 列)
検索拡張生成,5133,5133,6434,名詞,普通名詞,サ変可能,*,*,* ... (30 列)
参政権,5146,5146,4198,名詞,普通名詞,一般,*,*,* ... (30 列)
フガシ,4786,4786,3817,名詞,固有名詞,一般,*,*,* ... (30 列)
都立大,4786,4786,8591,名詞,固有名詞,一般,*,*,* ... (30 列)

前: 大 | 規模 | 言語 | モデル | で | 検索 | 拡張 | 生成 | を | 試す | 。
後: 大規模言語モデル | で | 検索拡張生成 | を | 試す | 。
前: 外国 | 人参 | 政権 | に | つい | て | 話し合っ | た | 。
後: 外国 | 人 | 参政権 | に | つい | て | 話し合っ | た | 。
前: 形態 | 素 | 解析 | に | フ | ガシ | を | 使う | 。
後: 形態 | 素 | 解析 | に | フガシ | を | 使う | 。
前: 都立 | 大 | に | 行く | 。
後: 都立大 | に | 行く | 。

形態	名詞	普通名詞	一般	ケイタイ	False
素	接尾辞	名詞的	一般	ソ	False
解析	名詞	普通名詞	サ変可能	カイセキ	False
に	助詞	格助詞	*	ニ	False
フガシ	名詞	固有名詞	一般	フガシ	False
を	助詞	格助詞	*	ヲ	False
使う	動詞	一般	*	ツカウ	False
。	補助記号	句点	*		False


In [4]:
# 「都立大」の生起コストだけを変えて作り直し、ほかの文の分割が崩れないかを見る（コストが小さいほど 1 語として選ばれやすい）
checks = ["都立大に行く。", "都立大の学生", "都立大学に行く。", "東京都立大学に行く。"]
ref_cost = int(ref_entry("都立大", "トリツダイ", "東大")[3])
for cost in [0, ref_cost, 11000, 14000]:
    t = build_userdic([ref_entry("都立大", "トリツダイ", "東大", cost)], f"toritsu_{cost}")
    print(f"{cost:>6}", " / ".join(" ".join(w.surface for w in t(s)) for s in checks))

     0 都立大 に 行く 。 / 都立大 の 学生 / 都立大 学 に 行く 。 / 東京 都立大 学 に 行く 。
  8591 都立大 に 行く 。 / 都立大 の 学生 / 都立大 学 に 行く 。 / 東京 都立 大学 に 行く 。
 11000 都立大 に 行く 。 / 都立大 の 学生 / 都立 大学 に 行く 。 / 東京 都立 大学 に 行く 。
 14000 都立 大 に 行く 。 / 都立 大 の 学生 / 都立 大学 に 行く 。 / 東京 都立 大学 に 行く 。


In [5]:
# 「都立大」のコストを 11000 にして辞書を作り直し、期待どおりに分割されるかを単体テスト（回帰テスト）で確かめる
rows = [ref_entry(*e) for e in entries[:-1]] + [ref_entry("都立大", "トリツダイ", "東大", 11000)]
user = build_userdic(rows, "user")
cases = [
    ("大規模言語モデルを作る", ["大規模言語モデル", "を", "作る"]),
    ("検索拡張生成で答える", ["検索拡張生成", "で", "答える"]),
    ("外国人参政権の議論", ["外国", "人", "参政権", "の", "議論"]),
    ("フガシで解析する", ["フガシ", "で", "解析", "する"]),
    ("都立大に行く", ["都立大", "に", "行く"]),
    ("都立大学に行く", ["都立", "大学", "に", "行く"]),          # 登録した語が他の語の一部を奪わない
    ("東京都立大学の言語モデル", ["東京", "都立", "大学", "の", "言語", "モデル"]),
]
for sent, expected in cases:
    got = [w.surface for w in user(sent)]
    assert got == expected, f"{sent}: {got}"
print(f"単体テスト: {len(cases)}/{len(cases)} 件合格")

単体テスト: 7/7 件合格


- MeCab のユーザー辞書の CSV は「表層形,左文脈 ID,右文脈 ID,生起コスト,素性…」の形で、素性の列はシステム辞書と同じ並び（unidic-lite は 26 列、IPA 辞書は 9 列）にする。左右文脈 ID は前後の語とのつながりやすさ（連接コスト）を引くための番号で、UniDic では品詞だけでなく語種でも分かれる（上の出力では、漢語の普通名詞「言語」が 5146、外来語の「モデル」が 5144、固有名詞の「東大」が 4786）。ID は辞書ごとに違うので、同じ品詞・語種の既存語から借りるのが確実で、システム辞書を変えたら取り直して作り直す。ID の列を空にすると、`mecab-dict-index` が辞書の `left-id.def`・`right-id.def`・`rewrite.def` を使って素性から自動で付ける（手元で試すと、上と同じ 4786 や 5146 が付いた）。
- 生起コストは小さいほどその語が選ばれやすい。長い複合語は、元の分割の語のコストと連接コストの合計に勝てばよいので、似た語のコストを借りれば多くの場合 1 語になる（「大規模言語モデル」「検索拡張生成」「参政権」）。一方、短い語はほかの語の一部を奪いやすい。上の例では、「東大」から借りたコスト（8591）でも「都立大学」が「都立大 / 学」に、0 では「東京都立大学」まで「東京 / 都立大 / 学」に崩れ、14000 では「都立大」自体が 1 語にならなかった。11000 なら 4 文とも期待どおりで、このように「変わってほしい文」と「変わってほしくない文」を並べて、両方を満たすコストを探す。`mecab-dict-index` には学習済みのモデルからコストを推定する `-m`・`-a` もあるが、unidic-lite にはモデルのファイルが含まれていない。
- 「都立大学生」のような文は「都立大 / 学生」とも「都立大学 / 生」とも読めて、辞書だけでは決まらない（手元で試すと、コスト 11000 では前者、12000 では後者になった）。語を足したら、ここの単体テストのように期待する分割を並べた文で回帰テストを続ける。
- `fugashi.build_dictionary` は fugashi の wheel に同梱の libmecab の `mecab-dict-index` をそのまま呼ぶ関数で、コマンドの `fugashi-build-dict` からも使える（引数は `mecab-dict-index` と同じ）。エラー時には Python の例外にならずプロセスが終わるので、ノートブックでは別プロセスで呼んでいる。apt の `mecab-utils` や MeCab のソースからのビルドは要らない（検証はコンテナの中で root にならずに行った）。
- 使い分け: 数語〜数千語を UniDic の単位と品詞体系のまま足したいならこのレシピ。表記ゆれも正規化形でまとめたいなら、この章の「Sudachi にユーザー辞書をコンパイルして追加する」（18 列の CSV と `sudachipy ubuild`、分割の単位 A/B/C も指定できる）。コンパイルせずに手早く試すなら「Janome にユーザー辞書を追加する」（表層形・品詞・読みの 3 列の簡易辞書）。ウェブ上の新語・固有名詞を大量にまとめて入れたいなら「NEologd 辞書で新語・固有名詞をまとまりとして解析する」（IPA 辞書に追加したシステム辞書。2020 年で更新が止まっている）。登録する候補は、この章の「辞書にない未知語を見つける」や「専門用語の候補を FLR と C-value でスコア付けして抽出する」で集められる。
- unidic-lite（UniDic 2.1.2）のライセンスは BSD・LGPL・GPL のいずれか。fugashi は MIT ライセンス。